In [0]:
from pyspark.sql import functions as F

MY_ID = "arzoo"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"
S   = f"workspace.capstone_{MY_ID}"

def land(name):
    df = (spark.read.option("header", True)
                    .option("inferSchema", False)   # keeps every column a string
                    .csv(f"{VOL}/raw/{name}"))
    return (df
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_row_hash", F.sha2(F.concat_ws("||",
            *[F.coalesce(F.col(c), F.lit("")) for c in df.columns]), 256)))

land("complaints").write.mode("overwrite").saveAsTable(f"{S}.bronze_complaints")
land("wards").write.mode("overwrite").saveAsTable(f"{S}.bronze_wards")

In [0]:
b = spark.table(f"{S}.bronze_complaints")
print("complaints:", b.count())                                   # 62,300
print("wards:", spark.table(f"{S}.bronze_wards").count())         # 40
print("closed_ts = 'NA':", b.filter("closed_ts = 'NA'").count())  # 1,200
print("raised in 1970:", b.filter("raised_ts LIKE '1970%'").count())  # 700
print("distinct ids:", b.select("complaint_id").distinct().count())   # 60,500
b.printSchema()   # every data column should say string

complaints: 62300
wards: 40
closed_ts = 'NA': 1200
raised in 1970: 700
distinct ids: 60500
root
 |-- complaint_id: string (nullable = true)
 |-- ward_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raised_ts: string (nullable = true)
 |-- closed_ts: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _row_hash: string (nullable = true)

